In [ ]:
# Get BORI <--> KMG <--> Dutt Title mapping

cross_edition__title_map = []

def get_title_numbers(line):
    ls = line.split('</div>')
    
    a = ls[0]
    ali = a.rfind('>')
    bori_ch_id = a[ali+1:]
    
    b = ls[1]
    bli = b.rfind('>')
    kmg_ch_id = b[bli+1:]
    
    c = ls[2]
    cli = c.rfind('>')
    dutt_ch_id = c[cli+1:]
    
    return([bori_ch_id, kmg_ch_id, dutt_ch_id])
    
with open('svat_html.txt', 'r') as file:
        for idx, line in enumerate(file):
            if line.startswith('<tr id='):
                cross_edition__title_map.append(get_title_numbers(line))
                # break
                
len(cross_edition__title_map)
cross_edition__title_map_df = pd.DataFrame(cross_edition__title_map, columns=['bori_chapter_ids', 'kmg_chapter_ids', 'dutt_chapter_names'])
cross_edition__title_map_df.to_csv('bori_kmg_dutt_chapter_map.csv', index=False)

In [ ]:
# extract google_trans from old jsons
old_data_path = "formatted_json_data_old"
book_jsons = os.listdir(old_data_path)

all_gtrans_data = []
for bk in book_jsons:
    bk_path = os.path.join(old_data_path, bk)
    
    with open(bk_path, 'r', encoding='utf-8') as f:
        bk_obj = json.load(f)
        book_id = str(bk_obj['book_number']).zfill(2)
        for c in bk_obj['chapters']:
            chapter_id = c['chapter_name'][-3:]
            for v in c['verses']:
                verse_id = v['verse_number']
                verse_uid = book_id+chapter_id+verse_id
                v_sans = '। '.join(v['verse_data']) + '।'
                v_gtrans = v['verse_translation']['google_trans']
                
                all_gtrans_data.append([verse_uid, v_sans, v_gtrans])
                

# all_bori_gtrans_data_df = pd.DataFrame(all_gtrans_data, columns=['bori_id', 'sans', 'gtrans'])
# all_bori_gtrans_data_df.to_csv('all_bori_gtrans_data.csv', index=False)

all_gtrans_data_no_headings = []
heading_row = []
for row in all_gtrans_data:
    v_id = row[0]
    
    if v_id.endswith('h'):
        heading_row = row.copy()
    else:
        if heading_row:
            extended_sans = heading_row[1] + ' ' + row[1]
            extended_gtrans = heading_row[2] + ': ' + row[2]
            new_row = [v_id, extended_sans, extended_gtrans]
            all_gtrans_data_no_headings.append(new_row)
            heading_row = []
        else:
            all_gtrans_data_no_headings.append(row)
            
all_bori_gtrans_nh_data_df = pd.DataFrame(all_gtrans_data_no_headings, columns=['bori_id', 'sans', 'google_trans'])
all_bori_gtrans_nh_data_df.to_csv('all_bori_gtrans_data.csv', index=False)

In [ ]:
# extract debroy_trans from old jsons
debroy_data_path = "temp_debroy_data"
book_jsons = os.listdir(debroy_data_path)

all_dtrans_data = []
for bk in book_jsons:
    bk_path = os.path.join(debroy_data_path, bk)
    
    with open(bk_path, 'r', encoding='utf-8') as f:
        bk_obj = json.load(f)
        book_id = str(bk_obj['book_number']).zfill(2)
        for c in bk_obj['chapters']:
            chapter_id = c['chapter_name'][-3:]
            for v in c['verses']:
                verse_id = v['verse_number']
                verse_uid = book_id+chapter_id+verse_id
                v_sans = '। '.join(v['verse_data']) + '।'
                v_dtrans = v['verse_translation']['debroy_trans']
                v_strans = v['verse_translation'].get('sarvamai_trans', '')
                all_dtrans_data.append([verse_uid, v_sans, v_dtrans, v_strans])
                

all_dtrans_data_no_headings = []
all_strans_data_no_headings = []
heading_row = []
for row in all_dtrans_data:
    v_id = row[0]
    
    if v_id.endswith('h'):
        heading_row = row.copy()
    else:
        if heading_row:
            extended_sans = heading_row[1] + ' ' + row[1]
            extended_dtrans = heading_row[2] + ': ' + row[2]
            extended_strans = heading_row[3] + ': ' + row[3]
            
            if extended_dtrans == ': ':
                extended_dtrans = ''
                extended_strans = ''
            new_row_d = [v_id, extended_sans, extended_dtrans]
            new_row_s = [v_id, extended_sans, extended_strans]
            all_dtrans_data_no_headings.append(new_row_d)
            all_strans_data_no_headings.append(new_row_s)
            heading_row = []
        else:
            new_row_d = [row[0], row[1], row[2]]
            new_row_s = [row[0], row[1], row[3]]

            all_dtrans_data_no_headings.append(new_row_d)
            all_strans_data_no_headings.append(new_row_s)
            
all_bori_dtrans_nh_data_df = pd.DataFrame(all_dtrans_data_no_headings, columns=['bori_id', 'sans', 'dtrans'])
all_bori_strans_nh_data_df = pd.DataFrame(all_strans_data_no_headings, columns=['bori_id', 'sans', 'strans'])
all_bori_dtrans_nh_data_df.to_csv('all_bori_dtrans_data.csv', index=False)
all_bori_strans_nh_data_df.to_csv('all_bori_strans_data.csv', index=False)

In [ ]:
# building files for website
all_bori_dtrans_data = pd.read_csv('all_bori_dtrans_data.csv', dtype={'bori_id': object}).fillna('')
all_bori_dtrans_data = all_bori_dtrans_data.drop(columns=['sans'])
bori_debroy_dict = dict(sorted(all_bori_dtrans_data.values.tolist()))

all_bori_sarvamtrans_data = pd.read_csv('all_bori_strans_data.csv', dtype={'bori_id': object}).fillna('')
all_bori_sarvamtrans_data = all_bori_sarvamtrans_data.drop(columns=['sans'])
bori_sarvam_dict = dict(sorted(all_bori_sarvamtrans_data.values.tolist()))

# Create formatted jsons for website
import json
from collections import OrderedDict

for book_id in all_bori_gtrans_data_df['book_id'].unique():
    # print(book_id, book_id_dict[book_id])
    book_name = book_id_dict[book_id]
    book_df = all_bori_gtrans_data_df[all_bori_gtrans_data_df['book_id'] == book_id]
    
    book_obj = OrderedDict()
    book_obj["book_name"] = f'{book_name}'
    book_obj["book_number"] = int(book_id),
    book_obj["num_chapters"] = 0
    book_obj["chapters"] = []
    book_obj["book_total_verses"] = 0
    
    book_chapter_count = 0
    book_total_verses_count = 0
    book_chapters = []
    for chapter_id in book_df['chapter_id'].unique():
        book_chapter_count += 1
        chapter_df = book_df[book_df['chapter_id'] == chapter_id]
        chapter_name = f'Canto {chapter_id}'
        
        chapter_obj = OrderedDict()
        chapter_obj['chapter_id'] = chapter_id
        chapter_obj['chapter_name'] = chapter_name
        chapter_obj['verses'] = []
        chapter_obj['chapter_total_verses'] = 0
        
        chapter_total_verses = 0
        chapter_verses = []
        for v_ind, verse in chapter_df.iterrows():
            book_total_verses_count += 1
            chapter_total_verses += 1
            
            v_bori_id = verse['bori_id']
            v_sans = verse['sans']
            v_sans_list = v_sans.split('।')[:-1]
            v_sans_list = [vl.strip() for vl in v_sans_list]
            v_gtrans = verse['google_trans']
            v_dtrans = bori_debroy_dict[v_bori_id]
            v_strans = bori_sarvam_dict[v_bori_id]
            
            verse_obj = OrderedDict()
            verse_obj['verse_id'] = verse['verse_id']
            verse_obj['verse_sans_lines'] = v_sans_list
            verse_obj['verse_translation'] = OrderedDict()
            verse_obj['verse_translation']['debroy_trans'] = v_dtrans
            verse_obj['verse_translation']['google_trans'] = v_gtrans
            verse_obj['verse_translation']['sarvam_trans'] = v_strans
            
            chapter_verses.append(verse_obj)
        
        chapter_obj['verses'] = chapter_verses
        chapter_obj['chapter_total_verses'] = chapter_total_verses
        book_chapters.append(chapter_obj)
    
    book_obj['num_chapters'] = book_chapter_count
    book_obj['chapters'] = book_chapters
    book_obj['book_total_verses'] = book_total_verses_count
    
    
    book_savefile_name = os.path.join('website_data', f'{book_id}_{book_id_dict[book_id]}.json')
    with open(book_savefile_name, 'w', encoding='utf-8') as f:
        json.dump(book_obj, f, ensure_ascii=False, indent=4)
    print(f'Created {book_savefile_name}')
    